In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/fingerprint_mlp_fold0.pth
/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints/test_predicted_fingerprints.npy
/kaggle/input/datasets/eveliaveldrine/rdkit-wheels/numpy-2.5.3-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
/kaggle/input/datasets/eveliaveldrine/rdkit-wheels/pillow-12.3.0-cp313-cp313-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl
/kaggle/input/datasets/eveliaveldrine/rdkit-wheels/rdkit-2025.3.6-cp313-cp313-manylinux_2_28_x86_64.whl
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/sample_submission.csv
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/train.parquet
/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra/test.parquet


In [2]:
import os

# Carpeta contenedora donde está alojado el dataset en tu Input
WHEEL_DIR = '/kaggle/input/datasets/eveliaveldrine/rdkit-wheels/'

# Instalación buscando la rueda compatible en esa carpeta
!pip install --no-index --find-links={WHEEL_DIR} rdkit --quiet

In [3]:
import os
import gc
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from rdkit import Chem
from rdkit.Chem import rdFingerprintGenerator

# --- 1. RUTAS Y CONFIGURACIÓN ---
DATA_DIR = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
TEST_PATH = os.path.join(DATA_DIR, 'test.parquet')
TRAIN_PATH = os.path.join(DATA_DIR, 'train.parquet')

# Cambia este nombre al del dataset donde guardaste los pesos .pth de la Fase 2
MODEL_DIR = '/kaggle/input/datasets/ernestomiguez/casmi2026-phase2-fingerprints'
MODEL_PATH = os.path.join(MODEL_DIR, 'fingerprint_mlp_fold0.pth')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# --- 2. DEFINICIÓN DE LA RED Y DATASET (input_dim=2001) ---
class SpectrumMLP(nn.Module):
    def __init__(self, input_dim=2001, output_dim=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, 1024),
            nn.BatchNorm1d(1024),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(1024, output_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.net(x)

class MS2DatasetSanitized(Dataset):
    def __init__(self, df, bin_size=0.5, max_mz=1000.0):
        self.df = df.reset_index(drop=True)
        self.bin_size = bin_size
        self.max_mz = max_mz
        self.num_bins = int(max_mz / bin_size) # 2000 bins

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        mzs = row.get('ms2_mzs', [])
        ints = row.get('ms2_normalized_intensities', [])
        precursor_mz = row.get('precursor_mz', 0.0)
        
        spectrum_vector = np.zeros(self.num_bins, dtype=np.float32)
        
        if mzs is not None and ints is not None and len(mzs) > 0 and len(ints) > 0:
            mzs_arr = np.nan_to_num(np.array(mzs, dtype=np.float32), nan=0.0)
            ints_arr = np.nan_to_num(np.array(ints, dtype=np.float32), nan=0.0)
            
            valid_mask = (mzs_arr >= 0) & (mzs_arr < self.max_mz)
            mzs_arr = mzs_arr[valid_mask]
            ints_arr = ints_arr[valid_mask]
            
            if len(mzs_arr) > 0:
                bins = (mzs_arr / self.bin_size).astype(int)
                bins = np.clip(bins, 0, self.num_bins - 1)
                np.maximum.at(spectrum_vector, bins, ints_arr)
        
        # Concatena precursor_mz / 1000.0 para llegar exactamente a 2001 features
        x = np.append(spectrum_vector, np.float32(precursor_mz / 1000.0))
        return torch.tensor(x, dtype=torch.float32)

# --- 3. PASO 1: LEER TEST OCULTO E INFERIR FINGERPRINTS EN TIEMPO REAL ---
print("Cargando test.parquet...")
test_df = pd.read_parquet(TEST_PATH)

unique_mols = test_df['molecule_id'].unique()
print(f"Moléculas únicas detectadas en Test: {len(unique_mols):,}")

test_dataset = MS2DatasetSanitized(test_df, bin_size=0.5, max_mz=1000.0)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2)

print("Cargando pesos del modelo PyTorch...")
model = SpectrumMLP(input_dim=2001, output_dim=1024).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.eval()

pred_fps_list = []
with torch.no_grad():
    for x_batch in test_loader:
        x_batch = x_batch.to(DEVICE)
        preds = model(x_batch)
        pred_fps_list.append(preds.cpu().numpy())

test_fps = np.concatenate(pred_fps_list, axis=0)

# Liberar memoria de GPU
del model, test_loader, test_dataset
if torch.cuda.is_available():
    torch.cuda.empty_cache()
gc.collect()

# Promediar fingerprints y precursor_mz por molecule_id
test_df['fp_idx'] = np.arange(len(test_df))
mol_to_fp = {}
mol_to_mz = {}

for mol_id, group in test_df.groupby('molecule_id'):
    indices = group['fp_idx'].values
    avg_fp = test_fps[indices].mean(axis=0)
    mol_to_fp[mol_id] = (avg_fp > 0.5).astype(np.uint8)
    mol_to_mz[mol_id] = group['precursor_mz'].iloc[0]

del test_fps
gc.collect()

# --- 4. PASO 2: CONSTRUCCIÓN DE CANDIDATOS CON PRECURSOR MZ ---
print("Cargando librería de candidatos con masa precursora...")
train_df = pd.read_parquet(TRAIN_PATH)

col_smiles = 'normalized_smiles' if 'normalized_smiles' in train_df.columns else 'smiles'
candidate_df = train_df.drop_duplicates(subset=['inchikey14']).reset_index(drop=True)
del train_df
gc.collect()

morgan_gen = rdFingerprintGenerator.GetMorganGenerator(radius=2, fpSize=1024)
num_candidates = len(candidate_df)
candidate_fps_matrix = np.zeros((num_candidates, 1024), dtype=np.uint8)
candidate_mzs = np.zeros(num_candidates, dtype=np.float32)
valid_smiles = []

valid_count = 0
for idx, row in candidate_df.iterrows():
    smi = str(row[col_smiles]) if pd.notnull(row[col_smiles]) else ""
    if not smi:
        continue
    
    mol = Chem.MolFromSmiles(smi)
    if mol is not None:
        fp = morgan_gen.GetFingerprint(mol)
        candidate_fps_matrix[valid_count] = np.frombuffer(fp.ToBitString().encode('utf-8'), 'u1') - ord('0')
        candidate_mzs[valid_count] = row.get('precursor_mz', 0.0)
        valid_smiles.append(smi)
        valid_count += 1

candidate_fps_matrix = candidate_fps_matrix[:valid_count]
candidate_mzs = candidate_mzs[:valid_count]
candidate_df = pd.DataFrame({'smiles': valid_smiles, 'precursor_mz': candidate_mzs})

# SMILES de respaldo para fallback (los más frecuentes)
top_fallback_smiles = candidate_df['smiles'].head(25).tolist()

# --- 5. PASO 3: BÚSQUEDA TANIMOTO FILTRADA POR MASA (MZ) ---
print("Ejecutando la búsqueda Tanimoto con filtro de masa ±0.5 Da...")

def batch_tanimoto_similarity(query_fp, candidate_matrix):
    cand_sums = candidate_matrix.sum(axis=1)
    query_sum = query_fp.sum()
    intersection = np.dot(candidate_matrix, query_fp)
    union = cand_sums + query_sum - intersection
    scores = np.divide(intersection, union, out=np.zeros_like(intersection, dtype=float), where=union!=0)
    return np.nan_to_num(scores, nan=0.0)

results = []
top_k = 25
mz_tolerance = 0.5  # Ventana de masa en Daltons (±0.5 Da)

for mol_id in unique_mols:
    pred_bit_array = mol_to_fp[mol_id]
    target_mz = mol_to_mz.get(mol_id, 0.0)
    
    # Filtrar candidatos dentro de la ventana de masa ±0.5 Da
    mz_mask = np.abs(candidate_mzs - target_mz) <= mz_tolerance
    filtered_indices = np.where(mz_mask)[0]
    
    # Si la ventana es muy estrecha y hay menos de 25 candidatos, ampliar la tolerancia a ±2.0 Da
    if len(filtered_indices) < top_k:
        mz_mask = np.abs(candidate_mzs - target_mz) <= 2.0
        filtered_indices = np.where(mz_mask)[0]
    
    # Si aún hay suficientes candidatos filtrados
    if len(filtered_indices) >= top_k:
        sub_matrix = candidate_fps_matrix[filtered_indices]
        scores = batch_tanimoto_similarity(pred_bit_array, sub_matrix)
        
        top_sub_indices = np.argpartition(scores, -top_k)[-top_k:]
        top_sub_indices = top_sub_indices[np.argsort(scores[top_sub_indices])[::-1]]
        
        selected_smiles = candidate_df.iloc[filtered_indices[top_sub_indices]]['smiles'].tolist()
    elif len(filtered_indices) > 0:
        # Usar los que haya y rellenar con fallback
        sub_matrix = candidate_fps_matrix[filtered_indices]
        scores = batch_tanimoto_similarity(pred_bit_array, sub_matrix)
        sorted_sub_indices = np.argsort(scores)[::-1]
        selected_smiles = candidate_df.iloc[filtered_indices[sorted_sub_indices]]['smiles'].tolist()
        
        for fb_smi in top_fallback_smiles:
            if len(selected_smiles) >= top_k:
                break
            if fb_smi not in selected_smiles:
                selected_smiles.append(fb_smi)
    else:
        selected_smiles = top_fallback_smiles[:top_k]

    results.append({
        'molecule_id': mol_id,
        'smiles': ";".join(selected_smiles[:25])
    })

sub_df = pd.DataFrame(results)

# --- 6. GUARDAR Y VERIFICAR SUBMISSION.CSV ---
sub_df = sub_df[['molecule_id', 'smiles']]

assert len(sub_df) == len(unique_mols), "Número de filas no coincide con test"
assert not sub_df.isnull().values.any(), "Hay valores nulos en el submission"
assert sub_df['molecule_id'].nunique() == len(sub_df), "Hay IDs duplicados"

SUBMISSION_FILE = 'submission.csv'
sub_df.to_csv(SUBMISSION_FILE, index=False)

print(f"✓ Archivo '{SUBMISSION_FILE}' generado con éxito ({len(sub_df):,} filas)!")

Cargando test.parquet...
Moléculas únicas detectadas en Test: 400
Cargando pesos del modelo PyTorch...
Cargando librería de candidatos con masa precursora...
Ejecutando la búsqueda Tanimoto con filtro de masa ±0.5 Da...
✓ Archivo 'submission.csv' generado con éxito (400 filas)!
